# 🎲 終末地卡池數據模擬器 (Simulation Module)

歡迎使用本模組！本檔案程式碼較長，推薦使用 `Ctrl + F` 搜尋下方【模式專用關鍵字】，或直接點開 VS Code 左下角的 **OUTLINE（大綱）** 面板進行快速跳轉，閱讀體驗最佳。

---

## 🔍 快速尋找關鍵字清單
* 🛠️ 初始化設定：`0. 基本套件匯入與全域參數設定`
* 🎯 模式一：`1. 角色池模擬—首次獲得限定角色`
* 👑 模式二：`2. 角色池模擬—滿潛限定角色`
* 📊 模式三：`3. 專題分析—武庫配額獲取與武器池抽數轉換`
* ⚔️ 模式四：`4. 武器池模擬—首次獲取機率提升武器`
* ⚙️ 模式五：`5. 武器池模擬—滿潛機率提升武器`

---

## 0. 基本套件匯入與全域參數設定

In [1]:
import numpy as np
import pandas as pd
import os

DATA_DIR = "sim_data"
trials = 1_000_000

if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)


## 1. 角色池模擬—首次獲得限定角色

In [ ]:
class get_one_UPcharacter:
    def __init__(self):
        # Base probability and pity parameters
        self.PROB_6_STAR = 0.008
        self.PROB_5_STAR = 0.08
        self.SOFT_PITY_START = 65
        self.HARD_PITY_6_STAR = 80
        self.HARD_PITY_5_STAR = 10
        self.BIG_PITY_LIMIT = 120
        
        # State counters (Main Pool)
        self.num_pulls_for_char = 0    # Total actual paid pulls consumed
        self.pity_6_star_count = 0     # Pulls since last 6-star character
        self.pity_5_star_count = 0     # Pulls since last 5-star character
        self.big_pity_counter = 0      # Progress towards the 120-pull big pity
        
        self.is_big_pity_active = True # 120-pull big pity, effective ONCE per pool
        self.has_obtained_up = False   
        
        # Inventory records
        self.inventory = {
            'up_6_star_char': 0,
            'non_6_star_char': 0,
            '5_star_char': 0,
            '4_star_char': 0
        }

    def _get_current_prob_6(self):
        """Calculate soft pity probability scaling based on current pity count"""
        current_pity = self.pity_6_star_count + 1
        if current_pity <= self.SOFT_PITY_START:
            return self.PROB_6_STAR
        return self.PROB_6_STAR + (current_pity - self.SOFT_PITY_START) * 0.05

    def pull(self):
        """Execute a single paid pull"""
        self.num_pulls_for_char += 1
        if self.is_big_pity_active:
            self.big_pity_counter += 1

        prob_6 = self._get_current_prob_6()
        
        # Pity conditions
        force_up_6 = (self.is_big_pity_active and self.big_pity_counter >= self.BIG_PITY_LIMIT)
        force_any_6 = (self.pity_6_star_count + 1 >= self.HARD_PITY_6_STAR)
        force_any_5 = (self.pity_5_star_count + 1 >= self.HARD_PITY_5_STAR)

        rand = np.random.random()

        # 1. Evaluate 6-star drop
        if rand < prob_6 or force_any_6 or force_up_6:
            self.pity_6_star_count = 0
            self.pity_5_star_count = 0
            
            is_up = False
            if force_up_6:
                is_up = True
            else:
                is_up = np.random.random() < 0.5
            
            if is_up:
                self.inventory['up_6_star_char'] += 1
                self.has_obtained_up = True
                self.is_big_pity_active = False
            else:
                self.inventory['non_6_star_char'] += 1
        
        # 2. Evaluate 5-star drop (Protected from interval stacking anomaly)
        elif rand < (self.PROB_6_STAR + self.PROB_5_STAR) or force_any_5:
            self.pity_5_star_count = 0
            self.pity_6_star_count += 1
            self.inventory['5_star_char'] += 1
        
        # 3. Evaluate 4-star drop
        else:
            self.pity_6_star_count += 1
            self.pity_5_star_count += 1
            self.inventory['4_star_char'] += 1

    def pull_free(self, free_counters):
        """
        Execute a single free pull with completely isolated base probabilities.
        Receives mutable dict to persist free pity across the 10-pull loop.
        """
        Temp_PROB_6_STAR = 0.008
        Temp_PROB_5_STAR = 0.08
        Temp_HARD_PITY_5_STAR = 10

        # Increment transient counters dedicated to the free pool
        free_counters['pity_6'] += 1
        free_counters['pity_5'] += 1

        # Check if the 5-star hard pity is triggered within the free pool
        force_any_5 = (free_counters['pity_5'] >= Temp_HARD_PITY_5_STAR)

        rand = np.random.random()
        
        # 1. Evaluate 6-star drop (Strictly uses base probability 0.8%)
        if rand < Temp_PROB_6_STAR:
            # Reset free pool counters upon hitting a 6-star
            free_counters['pity_6'] = 0
            free_counters['pity_5'] = 0

            Temp_is_up = np.random.random() < 0.5
            if Temp_is_up:
                self.inventory['up_6_star_char'] += 1
                self.has_obtained_up = True  # Target acquired, terminates the simulation
            else:
                self.inventory['non_6_star_char'] += 1
        
        # 2. Evaluate 5-star drop (Triggers on base 8% OR 10-pull free hard pity)
        elif rand < (Temp_PROB_6_STAR + Temp_PROB_5_STAR) or force_any_5:
            # Reset 5-star counter only for the free pool
            free_counters['pity_5'] = 0
            self.inventory['5_star_char'] += 1
            
        # 3. Evaluate 4-star drop
        else:
            self.inventory['4_star_char'] += 1

    def run(self):
        """Simulate a single player process until obtaining at least 1 copy of UP character"""
        while not self.has_obtained_up:
            # Check for the reward mechanism BEFORE the next paid pull is executed.
            # If the player has just completed exactly 30 paid pulls, trigger the free pool.
            if self.num_pulls_for_char == 30:
                # Create transient counters valid only within this specific 10-pull lifecycle
                free_pool_counters = {'pity_6': 0, 'pity_5': 0}
                
                # Unconditionally execute all 10 free pulls as a guaranteed bonus
                for _ in range(10):
                    self.pull_free(free_pool_counters)
                
                # After completing the free 10-pull, check if the player graduated
                if self.has_obtained_up:
                    break
            
            # Execute the standard paid pull (if pity is 29, this becomes the 30th pull)
            self.pull()
        
        return {
            'num_pulls_for_char': self.num_pulls_for_char,
            'up_6_star_char': self.inventory['up_6_star_char'],
            'non_6_star_char': self.inventory['non_6_star_char'],
            '5_star_char': self.inventory['5_star_char'],
            '4_star_char': self.inventory['4_star_char']
        }

# --- Execute 1 Million Simulations ---
def main_process(total_sims):
    results = []
    print(f"Generating {total_sims} simulation records for Single UP Character ...")
    
    for i in range(total_sims):
        sim = get_one_UPcharacter()
        results.append(sim.run())
        
        # Progress report every 100k simulations
        if (i + 1) % (total_sims // 10) == 0:
            print(f"Progress: {(i + 1) / total_sims * 100:.0f}%")
            
    df = pd.DataFrame(results)
    
    # Save to CSV
    save_path = f"{DATA_DIR}/df_get_UPcharacter.csv"
    df.to_csv(save_path, index=False)
    print(f"Done! Raw data saved to: {save_path}")

if __name__ == "__main__":
    main_process(trials)

## 2. 角色池模擬—滿潛限定角色

In [ ]:
class get_Full_Potential_UPcharacter:
    def __init__(self):
        # Base probability and pity parameters
        self.PROB_6_STAR = 0.008
        self.PROB_5_STAR = 0.08
        self.SOFT_PITY_START = 65
        self.HARD_PITY_6_STAR = 80
        self.HARD_PITY_5_STAR = 10
        self.BIG_PITY_LIMIT = 120
        
        # State counters (Main Pool)
        self.num_pulls_for_char = 0    # Total actual paid pulls consumed
        self.pity_6_star_count = 0     # Pulls since last 6-star character
        self.pity_5_star_count = 0     # Pulls since last 5-star character
        self.big_pity_counter = 0      # Progress towards the 120-pull big pity
        
        self.is_big_pity_active = True # 120-pull big pity, effective ONCE per pool lifecycle
        
        # Inventory records
        self.inventory = {
            'up_6_star_char': 0,
            'non_6_star_char': 0,
            '5_star_char': 0,
            '4_star_char': 0
        }

    def _get_current_prob_6(self):
        """Calculate soft pity probability scaling based on current pity count"""
        current_pity = self.pity_6_star_count + 1
        if current_pity <= self.SOFT_PITY_START:
            return self.PROB_6_STAR
        return self.PROB_6_STAR + (current_pity - self.SOFT_PITY_START) * 0.05

    def _should_terminate(self):
        """
        Unified Centralized Termination Criteria.
        [EASY TO MODIFY PLACE FOR FUTURE SIMULATIONS]
        """
        # Mode 1: Full Potential Termination (Target: 6 copies of UP character)
        return self.inventory['up_6_star_char'] >= 6
        
        # Mode 2: Specific Pull Limit Termination (Uncomment below to use in the future)
        # return self.num_pulls_for_char >= 120  # e.g., 120, 240, 360, ..., 1200

    def pull(self):
        """Execute a single paid pull"""
        self.num_pulls_for_char += 1
        if self.is_big_pity_active:
            self.big_pity_counter += 1

        prob_6 = self._get_current_prob_6()
        
        # Pity conditions
        force_up_6 = (self.is_big_pity_active and self.big_pity_counter >= self.BIG_PITY_LIMIT)
        force_any_6 = (self.pity_6_star_count + 1 >= self.HARD_PITY_6_STAR)
        force_any_5 = (self.pity_5_star_count + 1 >= self.HARD_PITY_5_STAR)

        rand = np.random.random()

        # 1. Evaluate 6-star drop
        if rand < prob_6 or force_any_6 or force_up_6:
            self.pity_6_star_count = 0
            self.pity_5_star_count = 0
            
            is_up = False
            if force_up_6:
                is_up = True
            else:
                is_up = np.random.random() < 0.5
            
            if is_up:
                self.inventory['up_6_star_char'] += 1
                self.is_big_pity_active = False # Big pity acquired and consumed in this pool
            else:
                self.inventory['non_6_star_char'] += 1
        
        # 2. Evaluate 5-star drop (Protected from interval stacking anomaly)
        elif rand < (self.PROB_6_STAR + self.PROB_5_STAR) or force_any_5:
            self.pity_5_star_count = 0
            self.pity_6_star_count += 1
            self.inventory['5_star_char'] += 1
        
        # 3. Evaluate 4-star drop
        else:
            self.pity_6_star_count += 1
            self.pity_5_star_count += 1
            self.inventory['4_star_char'] += 1

    def pull_free(self, free_counters):
        """
        Execute a single free pull with completely isolated base probabilities.
        Receives mutable dict to persist free pity across the 10-pull loop.
        """
        Temp_PROB_6_STAR = 0.008
        Temp_PROB_5_STAR = 0.08
        Temp_HARD_PITY_5_STAR = 10

        # Increment transient counters dedicated to the free pool
        free_counters['pity_6'] += 1
        free_counters['pity_5'] += 1

        # Check if the 5-star hard pity is triggered within the free pool
        force_any_5 = (free_counters['pity_5'] >= Temp_HARD_PITY_5_STAR)

        rand = np.random.random()
        
        # 1. Evaluate 6-star drop (Strictly uses base probability 0.8%)
        if rand < Temp_PROB_6_STAR:
            free_counters['pity_6'] = 0
            free_counters['pity_5'] = 0

            Temp_is_up = np.random.random() < 0.5
            if Temp_is_up:
                self.inventory['up_6_star_char'] += 1
            else:
                self.inventory['non_6_star_char'] += 1
        
        # 2. Evaluate 5-star drop (Triggers on base 8% OR 10-pull free hard pity)
        elif rand < (Temp_PROB_6_STAR + Temp_PROB_5_STAR) or force_any_5:
            free_counters['pity_5'] = 0
            self.inventory['5_star_char'] += 1
            
        # 3. Evaluate 4-star drop
        else:
            self.inventory['4_star_char'] += 1

    def run(self):
        """Simulate a player process until centralized termination criteria is met"""
        TOKEN_MILESTONES = {240, 480, 720, 960, 1200}
        
        while not self._should_terminate():
            # [Mechanism 1] Trigger free 10-pull exactly when paid pulls count reaches 30
            if self.num_pulls_for_char == 30:
                free_pool_counters = {'pity_6': 0, 'pity_5': 0}
                for _ in range(10):
                    self.pull_free(free_pool_counters)
                
                # Check immediately in case free pulls satisfied the termination criteria
                if self._should_terminate():
                    break

            # [Mechanism 2] Execute the standard paid pull
            self.pull()
            
            # [Mechanism 3] Unconditionally distribute token rewards right after the milestone pull
            if self.num_pulls_for_char in TOKEN_MILESTONES:
                self.inventory['up_6_star_char'] += 1
            
            # Centralized check for paid pull and token rewards combined
            if self._should_terminate():
                break
        
        return {
            'num_pulls_for_char': self.num_pulls_for_char,
            'up_6_star_char': self.inventory['up_6_star_char'],
            'non_6_star_char': self.inventory['non_6_star_char'],
            '5_star_char': self.inventory['5_star_char'],
            '4_star_char': self.inventory['4_star_char']
        }

# --- Execute Simulations ---
def main_process(total_sims):
    results = []
    print(f"Generating {total_sims} simulation records for Full Potential UP Character...")
    
    for i in range(total_sims):
        sim = get_Full_Potential_UPcharacter()
        results.append(sim.run())
        
        # Progress report every 10% milestone
        if (i + 1) % (total_sims // 10) == 0:
            print(f"Progress: {(i + 1) / total_sims * 100:.0f}%")
            
    df = pd.DataFrame(results)
    
    # Save to CSV
    save_path = f"{DATA_DIR}/df_get_Full_Potential_UPcharacter.csv"
    df.to_csv(save_path, index=False)
    print(f"Done! Raw data saved to: {save_path}")

if __name__ == "__main__":
    main_process(trials)

## 3. 專題分析—武庫配額獲取與武器池抽數轉換

> **說明**：作為從「角色」跨入「武器」的預算過渡橋樑，幫玩家在課金前定錨預算。

In [2]:
class Fixed_Pulls:
    def __init__(self, target_pulls):
        # Base probability and pity parameters
        self.PROB_6_STAR = 0.008
        self.PROB_5_STAR = 0.08
        self.SOFT_PITY_START = 65
        self.HARD_PITY_6_STAR = 80
        self.HARD_PITY_5_STAR = 10
        self.BIG_PITY_LIMIT = 120
        
        # Target limit for this specific simulation node
        self.target_pulls = target_pulls
        
        # State counters (Main Pool)
        self.num_pulls_for_char = 0    # Total actual paid pulls consumed
        self.pity_6_star_count = 0     # Pulls since last 6-star character
        self.pity_5_star_count = 0     # Pulls since last 5-star character
        self.big_pity_counter = 0      # Progress towards the 120-pull big pity
        
        self.is_big_pity_active = True # 120-pull big pity, effective ONCE per pool lifecycle
        
        # Inventory records
        self.inventory = {
            'up_6_star_char': 0,
            'non_6_star_char': 0,
            '5_star_char': 0,
            '4_star_char': 0
        }

    def _get_current_prob_6(self):
        """Calculate soft pity probability scaling based on current pity count"""
        current_pity = self.pity_6_star_count + 1
        if current_pity <= self.SOFT_PITY_START:
            return self.PROB_6_STAR
        return self.PROB_6_STAR + (current_pity - self.SOFT_PITY_START) * 0.05

    def _should_terminate(self):
        """
        Unified Centralized Termination Criteria.
        Mode 2: Specific Pull Limit Termination
        """
        return self.num_pulls_for_char >= self.target_pulls

    def pull(self):
        """Execute a single paid pull"""
        self.num_pulls_for_char += 1
        if self.is_big_pity_active:
            self.big_pity_counter += 1

        prob_6 = self._get_current_prob_6()
        
        # Pity conditions
        force_up_6 = (self.is_big_pity_active and self.big_pity_counter >= self.BIG_PITY_LIMIT)
        force_any_6 = (self.pity_6_star_count + 1 >= self.HARD_PITY_6_STAR)
        force_any_5 = (self.pity_5_star_count + 1 >= self.HARD_PITY_5_STAR)

        rand = np.random.random()

        # 1. Evaluate 6-star drop
        if rand < prob_6 or force_any_6 or force_up_6:
            self.pity_6_star_count = 0
            self.pity_5_star_count = 0
            
            is_up = False
            if force_up_6:
                is_up = True
            else:
                is_up = np.random.random() < 0.5
            
            if is_up:
                self.inventory['up_6_star_char'] += 1
                self.is_big_pity_active = False # Big pity acquired and consumed in this pool
            else:
                self.inventory['non_6_star_char'] += 1
        
        # 2. Evaluate 5-star drop (Protected from interval stacking anomaly)
        elif rand < (self.PROB_6_STAR + self.PROB_5_STAR) or force_any_5:
            self.pity_5_star_count = 0
            self.pity_6_star_count += 1
            self.inventory['5_star_char'] += 1
        
        # 3. Evaluate 4-star drop
        else:
            self.pity_6_star_count += 1
            self.pity_5_star_count += 1
            self.inventory['4_star_char'] += 1

    def pull_free(self, free_counters):
        """
        Execute a single free pull with completely isolated base probabilities.
        """
        Temp_PROB_6_STAR = 0.008
        Temp_PROB_5_STAR = 0.08
        Temp_HARD_PITY_5_STAR = 10

        free_counters['pity_6'] += 1
        free_counters['pity_5'] += 1

        force_any_5 = (free_counters['pity_5'] >= Temp_HARD_PITY_5_STAR)

        rand = np.random.random()
        
        # 1. Evaluate 6-star drop (Strictly uses base probability 0.8%)
        if rand < Temp_PROB_6_STAR:
            free_counters['pity_6'] = 0
            free_counters['pity_5'] = 0

            Temp_is_up = np.random.random() < 0.5
            if Temp_is_up:
                self.inventory['up_6_star_char'] += 1
            else:
                self.inventory['non_6_star_char'] += 1
        
        # 2. Evaluate 5-star drop (Triggers on base 8% OR 10-pull free hard pity)
        elif rand < (Temp_PROB_6_STAR + Temp_PROB_5_STAR) or force_any_5:
            free_counters['pity_5'] = 0
            self.inventory['5_star_char'] += 1
            
        # 3. Evaluate 4-star drop
        else:
            self.inventory['4_star_char'] += 1

    def run(self):
        """Simulate a player process until target pull budget is fully spent"""
        TOKEN_MILESTONES = {240, 480, 720, 960, 1200}
        
        while not self._should_terminate():
            # [Mechanism 1] Trigger free 10-pull exactly when paid pulls count reaches 30
            if self.num_pulls_for_char == 30:
                free_pool_counters = {'pity_6': 0, 'pity_5': 0}
                for _ in range(10):
                    self.pull_free(free_pool_counters)

            # [Mechanism 2] Execute the standard paid pull
            self.pull()
            
            # [Mechanism 3] Unconditionally distribute token rewards right after the milestone pull
            if self.num_pulls_for_char in TOKEN_MILESTONES:
                self.inventory['up_6_star_char'] += 1
            
            # Centralized check: Break immediately at the bottom to eliminate the final redundant while-loop re-evaluation
            if self._should_terminate():
                break
        
        return {
            'num_pulls_for_char': self.num_pulls_for_char,
            'up_6_star_char': self.inventory['up_6_star_char'],
            'non_6_star_char': self.inventory['non_6_star_char'],
            '5_star_char': self.inventory['5_star_char'],
            '4_star_char': self.inventory['4_star_char']
        }

# --- Execute Simulations Loop for All Budget Nodes ---
def main_process(total_sims):
    # pull_nodes = [80, 120, 240, 480, 720, 960]
    pull_nodes = [80, 120]
    
    for nodes in pull_nodes:
        results = []
        print(f"Generating {total_sims} simulation records for Fixed Pulls Node: {nodes}...")
        
        for i in range(total_sims):
            sim = Fixed_Pulls(target_pulls=nodes)
            results.append(sim.run())
            
            # Progress report every 10% milestone
            if (i + 1) % (total_sims // 10) == 0:
                print(f"Node {nodes} Pulls - Progress: {(i + 1) / total_sims * 100:.0f}%")
                
        df = pd.DataFrame(results)
        
        # --- Data Shaping and Redistribution Logic (Executed prior to CSV export) ---
        # Calculate the overflow amount of UP characters (values below 6 will precisely capture as 0)
        nums_exceed_6 = (df['up_6_star_char'] - 6).clip(lower=0)
        # Transfer the overflowed copies to the standard non-up pool category
        df['non_6_star_char'] += nums_exceed_6
        # Cap the standard UP character inventory at a maximum of 6 copies (Full Potential)
        df['up_6_star_char'] = df['up_6_star_char'].clip(upper=6)
        
        # Save to CSV
        save_path = f"{DATA_DIR}/df_{nodes}_pulls_for_weapon_quota.csv"
        df.to_csv(save_path, index=False)
        print(f"Done! Data for {nodes} pulls saved to: {save_path}\n")

if __name__ == "__main__":
    main_process(trials)

Generating 1000000 simulation records for Fixed Pulls Node: 80...
Node 80 Pulls - Progress: 10%
Node 80 Pulls - Progress: 20%
Node 80 Pulls - Progress: 30%
Node 80 Pulls - Progress: 40%
Node 80 Pulls - Progress: 50%
Node 80 Pulls - Progress: 60%
Node 80 Pulls - Progress: 70%
Node 80 Pulls - Progress: 80%
Node 80 Pulls - Progress: 90%
Node 80 Pulls - Progress: 100%
Done! Data for 80 pulls saved to: sim_data/df_80_pulls_for_weapon_quota.csv

Generating 1000000 simulation records for Fixed Pulls Node: 120...
Node 120 Pulls - Progress: 10%
Node 120 Pulls - Progress: 20%
Node 120 Pulls - Progress: 30%
Node 120 Pulls - Progress: 40%
Node 120 Pulls - Progress: 50%
Node 120 Pulls - Progress: 60%
Node 120 Pulls - Progress: 70%
Node 120 Pulls - Progress: 80%
Node 120 Pulls - Progress: 90%
Node 120 Pulls - Progress: 100%
Done! Data for 120 pulls saved to: sim_data/df_120_pulls_for_weapon_quota.csv



## 4. 武器池模擬—首次獲取機率提升武器

In [ ]:
class get_one_UPweapon:
    def __init__(self):
        # Base probabilities for a single item inside a multi-pull
        self.PROB_6_STAR = 0.04
        self.PROB_5_STAR = 0.15
        
        # State counters
        self.num_pulls_for_weapon = 0       # Total number of "multi-pulls" consumed (1, 2, 3...)
        self.pity_6_star_singles = 0        # CRITICAL: Relative single pulls elapsed since the last 6-star
        
        # Flag for the unique 8th multi-pull guarantee
        self.is_8th_multi_pity_active = True 
        self.blue_quota = 0
        
        self.inventory = {
            'up_6_star_weapon': 0,
            'non_up_6_star_weapon': 0,
            '5_star_weapon': 0,
            '4_star_weapon': 0
        }

    def _should_terminate(self):
        # First Copy Graduation (Terminates immediately when 1st UP is obtained)
        return self.inventory['up_6_star_weapon'] >= 1

    def _roll_six_star(self, is_forced_up=False):
        """Helper to process a 6-star acquisition from pulls (Awards 50 blue_quota)"""
        self.blue_quota += 50  
        # Whichever way a 6-star is obtained, the relative counter resets to 0 immediately!
        self.pity_6_star_singles = 0 
        
        is_up = is_forced_up or (np.random.random() < 0.25)
        if is_up:
            self.inventory['up_6_star_weapon'] += 1
            self.is_8th_multi_pity_active = False # Deactivate big pity forever if UP is obtained
        else:
            self.inventory['non_up_6_star_weapon'] += 1

    def execute_one_multi_pull(self):
        """Execute exactly ONE multi-pull batch (10 slots) with real-time rolling counters"""
        self.num_pulls_for_weapon += 1
        
        # Local trackers to monitor drops within this specific multi-pull batch
        six_stars_in_this_multi = 0
        five_stars_in_this_multi = 0
        
        # Process a 10-slot loop where each slot represents a real-time single pull sequence
        for slot in range(10):
            is_last_slot = (slot == 9)
            
            # --- Establish Real-Time Pity Conditions ---
            # 1. Big Pity: Absolute 80th pull (Last slot of 8th multi)
            force_up_6 = is_last_slot and self.is_8th_multi_pity_active and (self.num_pulls_for_weapon == 8)
            
            # 2. Regular Pity: Real-time 40th rolling single pull without any 6-star weapon
            force_any_6 = (self.pity_6_star_singles + 1 == 40)
            
            # Status flag to check if the current single pull yields a 6-star
            slot_yielded_6_star = False
            
            # --- Core Priority Interception (Evaluated dynamically per single pull) ---
            if force_up_6:
                self._roll_six_star(is_forced_up=True)
                six_stars_in_this_multi += 1
                slot_yielded_6_star = True
                
            elif force_any_6:
                # Real-time regular pity intercept: Triggers immediately at the exact 40th pull
                self._roll_six_star(is_forced_up=False)
                six_stars_in_this_multi += 1
                slot_yielded_6_star = True
                
            else:
                # Standard Probability Roll for the current slot
                rand = np.random.random()
                
                # A: Check if 6-star is rolled naturally (4% chance)
                if rand < self.PROB_6_STAR:
                    self._roll_six_star(is_forced_up=False)
                    six_stars_in_this_multi += 1
                    slot_yielded_6_star = True
                
                # B: If 6-star is NOT rolled, check if 5-star is rolled naturally OR forced by 10-pull pity
                # Pity condition: Last slot AND no 5-star or 6-star was found in the previous 9 slots
                elif (rand < (self.PROB_6_STAR + self.PROB_5_STAR)) or \
                     (is_last_slot and (six_stars_in_this_multi == 0) and (five_stars_in_this_multi == 0)):
                    self.inventory['5_star_weapon'] += 1
                    self.blue_quota += 10
                    five_stars_in_this_multi += 1
                
                # C: Otherwise, it's a 4-star weapon
                else:
                    self.inventory['4_star_weapon'] += 1
            
            # --- Real-Time Counter Maintenance ---
            # Increment the rolling counter only if NO 6-star weapon was obtained in this specific single pull.
            if not slot_yielded_6_star:
                self.pity_6_star_singles += 1

    def run(self):
        """Simulate the weapon pool process per multi-pull unit until graduation"""
        while not self._should_terminate():
            # Step 1: Execute the standard multi-pull
            self.execute_one_multi_pull()
            
            # Step 2: Progress Bar Check - Directly gift items upon reaching specific milestones
            current_count = self.num_pulls_for_weapon
            
            # Pattern A: Non-up Choice Box at 10, 26, 42, 58, 74...
            if current_count == 10 or (current_count > 10 and (current_count - 10) % 16 == 0):
                self.inventory['non_up_6_star_weapon'] += 1
                
            # Pattern B: Up Weapon Milestone at 18, 34, 50, 66, 82...
            if current_count == 18 or (current_count > 18 and (current_count - 18) % 16 == 0):
                self.inventory['up_6_star_weapon'] += 1
                
            # Step 3: Centralized check immediately after all drops and gifts are resolved
            if self._should_terminate():
                break
            
        return {
            'num_pulls_for_weapon': self.num_pulls_for_weapon,
            'up_6_star_weapon': self.inventory['up_6_star_weapon'],
            'non_up_6_star_weapon': self.inventory['non_up_6_star_weapon'],
            '5_star_weapon': self.inventory['5_star_weapon'],
            '4_star_weapon': self.inventory['4_star_weapon'],
            'blue_quota': self.blue_quota
        }
    
# --- Execute Simulations ---
def main_process(total_sims):
    results = []
    print(f"Generating {total_sims} simulation records for single UP Weapon ...")
    
    for i in range(total_sims):
        sim = get_one_UPweapon()
        results.append(sim.run())
        
        if (i + 1) % (total_sims // 10) == 0:
            print(f"Progress: {(i + 1) / total_sims * 100:.0f}%")
            
    df = pd.DataFrame(results)
    save_path = f"{DATA_DIR}/df_get_one_UPweapon.csv"
    df.to_csv(save_path, index=False)
    print(f"Done! Raw data saved to: {save_path}")

if __name__ == "__main__":
    main_process(trials)


## 5. 武器池模擬—滿潛機率提升武器

In [ ]:
class get_Full_Potential_UPweapon:
    def __init__(self):
        self.PROB_6_STAR = 0.04
        self.PROB_5_STAR = 0.15
        
        # State counters
        self.num_pulls_for_weapon = 0       # Total number of "multi-pulls" consumed (1, 2, 3...)
        self.pity_6_star_singles = 0        # CRITICAL: Relative single pulls elapsed since the last 6-star
        
        self.is_8th_multi_pity_active = True 
        self.blue_quota = 0
        
        self.inventory = {
            'up_6_star_weapon': 0,
            'non_up_6_star_weapon': 0,
            '5_star_weapon': 0,
            '4_star_weapon': 0
        }

    def _should_terminate(self):
        # Full Potential Graduation Criterion (up_6_star_weapon >= 6)
        return self.inventory['up_6_star_weapon'] >= 6

    def _roll_six_star(self, is_forced_up=False):
        """Helper to process a 6-star acquisition from pulls (Awards 50 blue_quota)"""
        self.blue_quota += 50  
        # Whichever way a 6-star is obtained, the relative counter resets to 0 immediately!
        self.pity_6_star_singles = 0 
        
        is_up = is_forced_up or (np.random.random() < 0.25)
        if is_up:
            self.inventory['up_6_star_weapon'] += 1
            self.is_8th_multi_pity_active = False 
        else:
            self.inventory['non_up_6_star_weapon'] += 1

    def execute_one_multi_pull(self):
        """Execute exactly ONE multi-pull batch (10 slots) with real-time rolling counters"""
        self.num_pulls_for_weapon += 1
        
        # Local trackers to monitor drops within this specific multi-pull batch
        six_stars_in_this_multi = 0
        five_stars_in_this_multi = 0
        
        # Process a 10-slot loop where each slot represents a real-time single pull sequence
        for slot in range(10):
            is_last_slot = (slot == 9)
            
            # --- Establish Real-Time Pity Conditions ---
            # 1. Big Pity: Absolute 80th pull (Last slot of 8th multi)
            force_up_6 = is_last_slot and self.is_8th_multi_pity_active and (self.num_pulls_for_weapon == 8)
            
            # 2. Regular Pity: Real-time 40th rolling single pull without any 6-star weapon
            force_any_6 = (self.pity_6_star_singles + 1 == 40)
            
            # Status flag to check if the current single pull yields a 6-star
            slot_yielded_6_star = False
            
            # --- Core Priority Interception (Evaluated dynamically per single pull) ---
            if force_up_6:
                self._roll_six_star(is_forced_up=True)
                six_stars_in_this_multi += 1
                slot_yielded_6_star = True
                
            elif force_any_6:
                # Real-time regular pity intercept: Triggers immediately at the exact 40th pull
                self._roll_six_star(is_forced_up=False)
                six_stars_in_this_multi += 1
                slot_yielded_6_star = True
                
            else:
                # Standard Probability Roll for the current slot
                rand = np.random.random()
                
                # A: Check if 6-star is rolled naturally (4% chance)
                if rand < self.PROB_6_STAR:
                    self._roll_six_star(is_forced_up=False)
                    six_stars_in_this_multi += 1
                    slot_yielded_6_star = True
                
                # B: If 6-star is NOT rolled, check if 5-star is rolled naturally OR forced by 10-pull pity
                # Pity condition: Last slot AND no 5-star or 6-star was found in the previous 9 slots
                elif (rand < (self.PROB_6_STAR + self.PROB_5_STAR)) or \
                     (is_last_slot and (six_stars_in_this_multi == 0) and (five_stars_in_this_multi == 0)):
                    self.inventory['5_star_weapon'] += 1
                    self.blue_quota += 10
                    five_stars_in_this_multi += 1
                
                # C: Otherwise, it's a 4-star weapon
                else:
                    self.inventory['4_star_weapon'] += 1
            
            # --- Real-Time Counter Maintenance ---
            # Increment the rolling counter only if NO 6-star weapon was obtained in this specific single pull.
            if not slot_yielded_6_star:
                self.pity_6_star_singles += 1

    def run(self):
        while not self._should_terminate():
            self.execute_one_multi_pull()
            
            current_count = self.num_pulls_for_weapon
            
            # Pattern A: Non-up Choice Box at 10, 26, 42, 58, 74
            if current_count == 10 or (current_count > 10 and (current_count - 10) % 16 == 0):
                self.inventory['non_up_6_star_weapon'] += 1
                
            # Pattern B: Up Weapon Milestone at 18, 34, 50, 66, 82
            if current_count == 18 or (current_count > 18 and (current_count - 18) % 16 == 0):
                self.inventory['up_6_star_weapon'] += 1
                
            if self._should_terminate():
                break
            
        return {
            'num_pulls_for_weapon': self.num_pulls_for_weapon,
            'up_6_star_weapon': self.inventory['up_6_star_weapon'],
            'non_up_6_star_weapon': self.inventory['non_up_6_star_weapon'],
            '5_star_weapon': self.inventory['5_star_weapon'],
            '4_star_weapon': self.inventory['4_star_weapon'],
            'blue_quota': self.blue_quota
        }

# --- Execute Simulations ---
def main_process(total_sims):
    results = []
    print(f"Generating {total_sims} simulation records for Full Potential UP Weapon ...")
    
    for i in range(total_sims):
        sim = get_Full_Potential_UPweapon()
        results.append(sim.run())
        
        if (i + 1) % (total_sims // 10) == 0:
            print(f"Progress: {(i + 1) / total_sims * 100:.0f}%")
            
    df = pd.DataFrame(results)
    save_path = f"{DATA_DIR}/df_get_Full_Potential_UPweapon.csv"
    df.to_csv(save_path, index=False)
    print(f"Done! Raw data saved to: {save_path}")

if __name__ == "__main__":
    main_process(trials)